# Notebook 02: Activation Patching & Circuit Verification

**Goal:** Verify the causal role of induction heads.

**Outputs:** `experiments/results/baseline_attribution_scores.npz`, `paper/figures/fig1_*`, `paper/figures/fig4_*`

**Runtime:** ~15 minutes on CPU.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path
from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, set_global_seed
from src.circuits.patching import compute_circuit_attribution, get_circuit_heads
from src.viz.circuit_diagram import plot_circuit_diagram, plot_attribution_heatmap

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
model_config = ModelConfig(); eval_config = EvalConfig()
model = load_pretrained_model(model_config, device=device)

In [ ]:
attribution_scores = compute_circuit_attribution(
    model=model, seq_len=20, batch_size=32, seed=eval_config.seed, device=device,
)
circuit_heads = get_circuit_heads(attribution_scores, threshold=eval_config.induction_circuit_threshold)
print('Circuit heads:', circuit_heads)
print('Max attribution:', attribution_scores.max().item())

In [ ]:
results_dir = Path('../experiments/results'); results_dir.mkdir(parents=True, exist_ok=True)
np.savez(results_dir / 'baseline_attribution_scores.npz',
         attribution_scores=attribution_scores.cpu().numpy(),
         circuit_heads=np.array(circuit_heads))
print('Saved.')

In [ ]:
figures_dir = Path('../paper/figures'); figures_dir.mkdir(parents=True, exist_ok=True)
fig1 = plot_circuit_diagram(circuit_heads=circuit_heads, n_layers=model.cfg.n_layers, n_heads=model.cfg.n_heads,
    title='Induction circuit (pretrained)', save_path=figures_dir / 'fig1_circuit_diagram')
plt.show()
fig4 = plot_attribution_heatmap(attribution_scores.cpu().numpy(), threshold=eval_config.induction_circuit_threshold,
    title='Attribution scores (pretrained)', save_path=figures_dir / 'fig4_attribution_baseline')
plt.show()